# multi_1h_strat_029

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (209).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 12 |
| Detected functions | resample_1hr_custom, compute_lhp_dsl_indicator, process_ticker, backtest_trades, calculate_metrics, apply_strategy |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Indices_Data_merged.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['DEUIDXEUR60', 'GBRIDXGBP60', 'USA30IDXUSD60', 'USA500IDXUSD60',
       'USATECHIDXUSD60']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

def compute_lhp_dsl_indicator(df, period=72, fastend=0.3, slowend=0.03):
    close = df['close'].values
    length = len(close)

    # === 1. LHP calculation ===
    prev_close = np.roll(close, 1)
    prev_close[0] = np.nan  # avoid log(0)
    with np.errstate(divide='ignore', invalid='ignore'):
        lhp = np.where((close != 0) & (prev_close != 0), np.log(np.abs(prev_close / close)), np.nan)

    # === 2. KAMA smoothing on LHP ===
    kama = np.empty(length)
    kama[:] = np.nan
    kama[0] = lhp[0] if not np.isnan(lhp[0]) else 0

    # Precompute rolling volatility (sum of abs diff of lhp)
    lhp_diff = np.abs(np.diff(lhp, prepend=np.nan))
    vol = pd.Series(lhp_diff).rolling(10, min_periods=1).sum().values

    for i in range(1, length):
        if np.isnan(lhp[i]) or np.isnan(lhp[i - 1]):
            kama[i] = kama[i - 1]
            continue
        er = np.abs(lhp[i] - lhp[i - 1]) / (vol[i] if vol[i] != 0 else 1e-10)
        sc = er * (fastend - slowend) + slowend
        kama[i] = kama[i - 1] + sc * (lhp[i] - kama[i - 1])

    # === 3. DSL Level calculation ===
    kama_series = pd.Series(kama)
    lhp_series = pd.Series(lhp)
    levelu = kama_series.where(lhp_series > kama_series).ffill()
    leveld = kama_series.where(lhp_series < kama_series).ffill()

    # === 4. Signal generation ===
    signal = np.zeros(length, dtype=int)
    signal[lhp > levelu] = 1
    signal[lhp < leveld] = 2

    # === 5. Output ===
    df['Signal'] = signal

    return df

def process_ticker(ticker, ticker_data, fee, slippage_rate, tp_pct, sl_pct):
    """Process one ticker's trades given its price/indicator data."""
    open_positions = {}
    trades = []
    cumulative_pnl = 0.0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = (
            row["date"],
            row["close"],
            row["open"],
            row["high"],
            row["low"],
            row["Signal"],
        )

        # Check existing position
        if ticker in open_positions:
            pos = open_positions[ticker]
            entry_price, direction, qty, entry_time = (
                pos["entry_price"],
                pos["direction"],
                pos["qty"],
                pos["entry_time"],
            )

            # Calculate target and stop levels
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            exit_price, exit_reason = None, None
            if direction == "long":
                if low <= sl_price and high >= tp_price:
                    exit_price, exit_reason = sl_price, "Both Hit (SL preferred)"
                elif low <= sl_price:
                    exit_price, exit_reason = sl_price, "Stop Loss"
                elif high >= tp_price:
                    exit_price, exit_reason = tp_price, "Take Profit"
            else:  # short
                if high >= sl_price and low <= tp_price:
                    exit_price, exit_reason = sl_price, "Both Hit (SL preferred)"
                elif high >= sl_price:
                    exit_price, exit_reason = sl_price, "Stop Loss"
                elif low <= tp_price:
                    exit_price, exit_reason = tp_price, "Take Profit"

            # Exit the trade if triggered
            if exit_price is not None:
                # Slippage as a proportion of the price
                slippage_amount = exit_price * slippage_rate
                exit_price = exit_price - slippage_amount if direction == "long" else exit_price + slippage_amount

                # Fees on both sides
                fee_cost = fee * (entry_price * qty) + fee * (exit_price * qty)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - fee_cost
                cumulative_pnl += pnl

                trades.append(
                    [
                        ticker,
                        entry_time,
                        timestamp,
                        entry_price,
                        exit_price,
                        pnl,
                        exit_reason,
                        cumulative_pnl,
                        direction,
                    ]
                )
                del open_positions[ticker]

        # Check new position entry
        if ticker not in open_positions:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            # Apply slippage on entry
            slippage_amount = open_price * slippage_rate
            entry_price = open_price + slippage_amount if direction == "long" else open_price - slippage_amount
            qty = 1000000 / entry_price  # position sizing

            open_positions[ticker] = {
                "entry_time": timestamp,
                "entry_price": entry_price,
                "direction": direction,
                "qty": qty,
            }

    return trades


def backtest_trades(data, tp_pct, sl_pct, slippage_rate=0.00002, fee=0.00004):
    """Backtest all tickers with given parameters."""
    tickers = data['Ticker'].unique()

    dtype = [
        ("Ticker", "U20"), ("date", "M8[ms]"), ("close", "f8"), ("open", "f8"),
        ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(
            data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
            .itertuples(index=False, name=None)
        ),
        dtype=dtype,
    )

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(ticker, data_np[data_np['Ticker'] == ticker], fee, slippage_rate, tp_pct, sl_pct)
        for ticker in tqdm(tickers)
    )

    trades = [trade for result in results for trade in result]

    return pd.DataFrame(
        trades,
        columns=[
            "Ticker",
            "Entry Time",
            "Exit Time",
            "Entry Price",
            "Exit Price",
            "PnL",
            "Exit Reason",
            "Cumulative PnL",
            "Direction",
        ],
    )


# === Usage Example ===
# Assume `data` is loaded and contains columns:
# ['Ticker', 'Date', 'open', 'high', 'low', 'close', 'Signal']

data['Date'] = pd.to_datetime(data['Date'])

# Compute your signals with a user-defined function
data = data.groupby('Ticker', group_keys=False).apply(compute_lhp_dsl_indicator)

# Avoid lookahead
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data.dropna(subset=['Signal'], inplace=True)
data['Signal'] = data['Signal'].astype(int)

# Run backtest
tradebook = backtest_trades(data, tp_pct=0.0035, sl_pct=0.0004, slippage_rate=0.00002, fee=0.00004)

# Save to CSV
tradebook.to_csv("/content/drive/MyDrive/tradebook_INDICES_LTPKAMADSL.csv", index=False)

# Inspect the tradebook
tradebook


In [ ]:
summary = data['Signal'].value_counts().reset_index()
summary.columns = ['Signal', 'Count']
print("Number of unique exit reasons:", len(summary))
print(summary)

In [ ]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# === Metrics Calculation Function (with R² and Monthly Consistency) ===
def calculate_metrics(tradebook):
    if tradebook.empty:
        return 0, 0, 0, 0

    # Ensure datetime
    tradebook['Date'] = pd.to_datetime(tradebook['Entry Time'])  # Replace with correct column if different

    equity = tradebook['PnL'].cumsum()
    total_profit = equity.iloc[-1]

    roll_max = equity.cummax()
    drawdown = roll_max - equity
    max_drawdown = drawdown.max()

    if max_drawdown == 0:
        ratio = float('inf') if total_profit > 0 else 0
    else:
        ratio = total_profit / max_drawdown

    # R² for equity curve smoothness
    x = np.arange(len(equity)).reshape(-1, 1)
    y = equity.values.reshape(-1, 1)
    model = LinearRegression().fit(x, y)
    r2 = model.score(x, y)

    # Monthly return consistency metric
    tradebook['Month'] = tradebook['Date'].dt.to_period('M')
    monthly_returns = tradebook.groupby('Month')['PnL'].sum()
    if len(monthly_returns) < 2:
        monthly_consistency = 0
    else:
        monthly_mean = monthly_returns.mean()
        monthly_std = monthly_returns.std()
        monthly_consistency = monthly_mean / monthly_std if monthly_std > 0 else float('inf')

    return ratio, total_profit, r2, monthly_consistency

# === Hyperparameter Grid ===
param_grid = {
    'period': [34, 55, 72, 89],
    'fastend': [0.8, 0.6, 0.4, 0.3, 0.2, 0.1],
    'slowend': [0.05, 0.04, 0.03, 0.02],
    'tp_pct': [0.0025, 0.003, 0.0035, 0.004, 0.0045, 0.005],
    'sl_pct': [0.0004, 0.0005, 0.0006]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    print(f"\n=== [{i}/{len(param_combinations)}] Testing Parameters ===")
    for k, v in param.items():
        print(f"{k}: {v}")

    temp_data = data.copy()

    def apply_strategy(group):
        return compute_lhp_dsl_indicator(
            group,
            period=param['period'],
            fastend=param['fastend'],
            slowend=param['slowend'],
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        sl_pct=param['sl_pct'],
        tp_pct=param['tp_pct']
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency = calculate_metrics(tradebook)

    print(f"➡️ Profit-to-Drawdown Ratio: {ratio:.4f}")
    print(f"➡️ Total Profit: ₹{total_profit:,.2f}")
    print(f"➡️ Equity Curve R²: {r2:.4f}")
    print(f"➡️ Monthly Consistency Score: {monthly_consistency:.4f}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.1 +
    results_df['Equity R2'] * 0.6 +
    results_df['Monthly Consistency'] * 0.3
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())
